In [2]:
import json 
from pathlib import Path
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from metrics import (
    mean_average_precision_score,
    F1_score_from_sim,
    IoU_from_sim,
    IoU_q
)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
output_folder = Path("/media/eceo_scratch_haas001/results/GRAM/test/")
gt_json = Path("../zero_shot/GRAM/dataset/test/ground_truth.json")
with open(gt_json, "r") as f:
    gt_queries_videos = json.load(f)
queries = gt_queries_videos.keys()
sim_matrix_df = pd.read_csv(output_folder / "similarity_matrix.csv").set_index("video_id")
sim_matrix_df

,An animal engaged in any activity other than foraging.,An animal that is neither a red deer nor a roe deer.,An animal running.,An animal bathing.,A roe deer grazing.,An animal browsing.,An adult roe deer sniffing.,A juvenile red deer scratching its body.,A chamois trotting.,An adult male roe deer jumping.,...,An animal running while reacting to a camera.,An animal looking at a camera.,An animal reacting to a camera and then foraging.,"An animal foraging, then reacting to a camera, and then returning to foraging.",An animal reacting to a camera and then running away.,"An animal foraging, then reacting to a camera, and then running away.",An animal reacting to a camera in rainy or overcast weather.,An animal reacting to a camera in clear or sunny weather.,A single adult red deer foraging only.,An empty video.
video_id,,,,,,,,,,,,,,,,,,,,,
S1_C1_E100_V0251,-0.964568,-0.961597,-0.952266,-0.980965,-0.923261,-0.955397,-0.917335,-0.938462,-0.982236,-0.922038,...,-0.928584,-0.961291,-0.941749,-0.938528,-0.932865,-0.919049,-0.937883,-0.950716,-0.925142,-0.971212
S1_C1_E103_V0253,-0.951148,-0.941746,-0.945132,-0.981318,-0.907478,-0.939549,-0.882842,-0.894601,-0.978788,-0.903049,...,-0.906029,-0.940912,-0.903619,-0.899757,-0.903650,-0.878225,-0.936607,-0.945256,-0.903484,-0.978220
S1_C1_E103_V0254,-0.952355,-0.932866,-0.952997,-0.981669,-0.873903,-0.945276,-0.852483,-0.866660,-0.977661,-0.884220,...,-0.923407,-0.951035,-0.911055,-0.904689,-0.920603,-0.888999,-0.943775,-0.953489,-0.867109,-0.977110
S1_C1_E103_V0255,-0.948939,-0.932381,-0.952913,-0.977354,-0.866531,-0.943453,-0.849721,-0.866036,-0.979003,-0.884299,...,-0.922723,-0.948307,-0.907669,-0.900658,-0.919186,-0.886198,-0.941576,-0.950323,-0.864508,-0.968629
S1_C1_E103_V0256,-0.948559,-0.932052,-0.952589,-0.976928,-0.864149,-0.944096,-0.851098,-0.866564,-0.978467,-0.884846,...,-0.922127,-0.948372,-0.907211,-0.899488,-0.919087,-0.885128,-0.941727,-0.949407,-0.862256,-0.972184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
S3_C5_F719_V0158,-0.960674,-0.960770,-0.944474,-0.984108,-0.939287,-0.952636,-0.922327,-0.926022,-0.981058,-0.916575,...,-0.902114,-0.955633,-0.915680,-0.913919,-0.901925,-0.884471,-0.943960,-0.950835,-0.935103,-0.973035
S3_C5_F863_V0178,-0.957484,-0.956498,-0.952128,-0.983740,-0.926997,-0.955516,-0.908466,-0.916218,-0.980849,-0.921869,...,-0.900207,-0.953532,-0.910122,-0.906496,-0.903225,-0.881934,-0.938329,-0.941996,-0.925886,-0.975967
S3_C5_F863_V0179,-0.956349,-0.934005,-0.949077,-0.983671,-0.895845,-0.945407,-0.864377,-0.875700,-0.976425,-0.890753,...,-0.907169,-0.941338,-0.904968,-0.905062,-0.906887,-0.884456,-0.937118,-0.945468,-0.886179,-0.978204


# mAP

In [7]:
mAP, APs = mean_average_precision_score(gt_queries_videos, sim_matrix_df)
print(f"mAP queries: {mAP: .3f}")
sorted(APs.items(), key=lambda x:x[1], reverse=True)

mAP queries:  0.060


[('An animal engaged in any activity other than foraging.',
  0.20964543768445545),
 ('A juvenile red deer playing.', nan),
 ('An animal resting.', 0.3054147628219026),
 ('A red deer resting in rainy weather.', nan),
 ('Rainy weather.', 0.4896618124157301),
 ('A video of an animal lying down while resting.', 0.2666698194962562),
 ('An adult male red deer rubbing its antlers on the ground.',
  0.09674954726244209),
 ('An animal running.', 0.06544179132506622),
 ('An animal participating in courtship.', 0.06149107125491292),
 ('An adult red deer standing with its head up while participating in courtship.',
  0.05797362126761152),
 ('An adult red deer vocalizing while participating in courtship.',
  0.056734550693912075),
 ('A juvenile deer in clear or sunny weather.', 0.056105122721568895),
 ('An animal bathing.', 0.038990122774568904),
 ('A wolf foraging.', 0.029392712550607287),
 ('An animal running while foraging.', 0.025101463339484018),
 ('An adult red deer browsing.', 0.025),
 ('A 

# mIoU and F1-score (require threshold)

Find the optimal threshold per query on train set

In [8]:
gt_json_train = Path("../zero_shot/GRAM/dataset/train/ground_truth.json")
output_folder_train = Path("/media/eceo_scratch_haas001/results/GRAM/train")
with open(gt_json_train, "r") as f:
    gt_queries_videos_train = json.load(f)
queries = gt_queries_videos_train.keys()
sim_matrix_train_df = pd.read_csv(output_folder_train / "similarity_matrix.csv").set_index("video_id")
sim_matrix_train_df

,An animal engaged in any activity other than foraging.,An animal that is neither a red deer nor a roe deer.,An animal running.,An animal bathing.,A roe deer grazing.,An animal browsing.,An adult roe deer sniffing.,A juvenile red deer scratching its body.,A chamois trotting.,An adult male roe deer jumping.,...,An animal looking at a camera.,A wolf reacting to a camera.,An animal reacting to a camera and then foraging.,"An animal foraging, then reacting to a camera, and then returning to foraging.",An animal reacting to a camera and then running away.,"An animal foraging, then reacting to a camera, and then running away.",An animal reacting to a camera in rainy or overcast weather.,An animal reacting to a camera in clear or sunny weather.,A single adult red deer foraging only.,An empty video.
video_id,,,,,,,,,,,,,,,,,,,,,
S1_C1_E104_V0262,-0.951993,-0.935011,-0.960021,-0.982698,-0.875153,-0.947459,-0.852060,-0.870437,-0.981917,-0.886899,...,-0.950602,-0.905825,-0.906353,-0.900258,-0.921213,-0.886798,-0.942923,-0.951831,-0.866776,-0.970270
S1_C1_E104_V0263,-0.953082,-0.934151,-0.961185,-0.982707,-0.877804,-0.947804,-0.855765,-0.873045,-0.980278,-0.888945,...,-0.951613,-0.910986,-0.910429,-0.904818,-0.925260,-0.893114,-0.945003,-0.953304,-0.869040,-0.969564
S1_C1_E104_V0264,-0.950579,-0.932802,-0.957093,-0.979251,-0.875009,-0.947375,-0.853505,-0.869362,-0.979612,-0.888382,...,-0.950177,-0.908023,-0.907578,-0.900658,-0.919520,-0.886936,-0.943772,-0.952103,-0.864502,-0.972996
S1_C1_E104_V0265,-0.952335,-0.932717,-0.952534,-0.980313,-0.868524,-0.945267,-0.848227,-0.864393,-0.978001,-0.882863,...,-0.946906,-0.907029,-0.909311,-0.903257,-0.920518,-0.888197,-0.944092,-0.952803,-0.863225,-0.973457
S1_C1_E104_V0266,-0.953372,-0.935144,-0.952731,-0.978695,-0.870270,-0.947068,-0.851168,-0.864713,-0.980280,-0.883223,...,-0.948637,-0.905892,-0.910089,-0.904461,-0.916945,-0.887371,-0.944615,-0.952314,-0.864791,-0.973354
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
S3_C5_F746_V0161,-0.956441,-0.930662,-0.946200,-0.982033,-0.887821,-0.942825,-0.855630,-0.863984,-0.975855,-0.879179,...,-0.939019,-0.891121,-0.906302,-0.903035,-0.900977,-0.878704,-0.937924,-0.945136,-0.879239,-0.978723
S3_C5_F760_V0162,-0.974048,-0.980564,-0.968489,-0.990404,-0.943940,-0.985698,-0.946642,-0.954286,-0.988044,-0.937009,...,-0.983129,-0.947027,-0.964362,-0.956218,-0.960842,-0.945139,-0.961564,-0.967684,-0.940488,-0.969211
S3_C5_F869_V0182,-0.986137,-0.991208,-0.978387,-0.997409,-0.963795,-0.990765,-0.972850,-0.976405,-0.992647,-0.959840,...,-0.990823,-0.974650,-0.986808,-0.980218,-0.985338,-0.976407,-0.993493,-0.985221,-0.968040,-0.974376


In [10]:
thrs = np.arange(sim_matrix_train_df.values.min(), sim_matrix_train_df.values.max(), step=0.01)
print("Trying thresholds:", thrs)
best_t = dict.fromkeys(gt_queries_videos_train.keys())
for q in gt_queries_videos_train.keys():
    best_IoU = 0
    if q not in sim_matrix_train_df.columns:
        print(f"Query {q} has not been processed")
        continue
    gt_videos_q = list(gt_queries_videos_train[q]["videos"])
    for t in thrs:
        ass_videos_q = list(sim_matrix_train_df.index[sim_matrix_train_df[q] > t])
        iou = IoU_q(gt_videos_q, ass_videos_q)
        if not np.isnan(iou) and iou > best_IoU:
            best_IoU = iou
            best_t[q] = t
    
    if best_t[q] is None:
        print(f"No best threshold found for {q}, using median similarity score as threshold")
        best_t[q] = np.median(sim_matrix_train_df.values)

Trying thresholds: [-0.99985766 -0.98985766 -0.97985766 -0.96985766 -0.95985766 -0.94985766
 -0.93985766 -0.92985766 -0.91985766 -0.90985766 -0.89985766 -0.88985766
 -0.87985766 -0.86985766 -0.85985766 -0.84985766 -0.83985766 -0.82985766
 -0.81985766 -0.80985766 -0.79985766]


In [11]:
mF1, F1_queries = F1_score_from_sim(gt_queries_videos=gt_queries_videos, sim_matrix_df=sim_matrix_df, best_thresholds=best_t)
mIoU, IoU_queries = IoU_from_sim(gt_queries_videos=gt_queries_videos, sim_matrix_df=sim_matrix_df, best_thresholds=best_t)

In [12]:
print(f"F1-score (macro-avg.): {mF1:.2f}")
print(f"mean IoU: {mIoU:.2f}")

F1-score (macro-avg.): 0.04
mean IoU: 0.02


In [13]:
results_df = pd.concat([
    pd.DataFrame.from_dict(IoU_queries, orient="index", columns=["mIoU"]),
    pd.DataFrame.from_dict(F1_queries, orient="index", columns=["F1-score"])
], axis=1)

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 90)
results_df.sort_values("mIoU", ascending=False)

,mIoU,F1-score
A single adult red deer foraging only.,0.372654,0.542969
An animal engaged in any activity other than foraging.,0.204724,0.339869
An adult female red deer foraging and a juvenile red deer foraging.,0.134228,0.236686
A video of two red deer.,0.121795,0.217143
Rainy weather.,0.096059,0.175281
An animal being vigilant while the weather is clear or sunny.,0.093791,0.171498
A video of two or more animals.,0.092328,0.169048
A video of an individual sniffing while reacting to a camera.,0.090909,0.166667
A video of an animal lying down while resting.,0.075949,0.141176
A red deer foraging in sunny weather.,0.068627,0.128440
